# LeetCode 2678 — Number of Senior Citizens bằng Python

Link đề: https://leetcode.com/problems/number-of-senior-citizens/ · Độ khó: **Easy** · Chủ đề: Array, String

Skeleton LeetCode:

```python
class Solution:
    def countSeniors(self, details: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biết **đọc một chuỗi có định dạng cố định**: biết ký tự/đoạn nào ứng với số điện thoại, giới tính, tuổi, chỗ ngồi, bằng **chỉ số** và **cắt lát (slicing)**.
2. Hiểu rõ quy tắc cắt lát `s[a:b]` — **lấy từ `a` đến `b − 1`** — và vì sao tuổi nằm ở `d[11:13]` (hai ký tự, chỉ số `11` và `12`).
3. Chuyển chuỗi số thành số nguyên bằng `int(...)`, hiểu cách `int("07") == 7`, và biết **so sánh chuỗi theo thứ tự từ điển** có thể thay `int` khi mọi chuỗi cùng độ dài.
4. Viết 5 cách: vòng lặp đếm tay, `sum` với biểu thức sinh, ghép chữ số bằng phép tính, so sánh chuỗi `d[11:13] > "60"`, và tách thành phần bằng **regex** (`re`).
5. Tránh các lỗi: dùng sai cận cắt lát (`d[11:12]`, `d[10:12]`), dùng `>= 60` thay cho `> 60` (đề: "**strictly more than** 60"), so sánh chuỗi với số mà không đổi kiểu, và nhầm vị trí giới tính với tuổi.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `details` (chỉ số từ 0). Mỗi phần tử mô tả **một hành khách** bằng một chuỗi **độ dài đúng `15`** theo quy ước:

- **10 ký tự đầu**: số điện thoại của hành khách.
- **Ký tự kế tiếp** (1 ký tự): giới tính (`M`, `F` hoặc `O`).
- **2 ký tự tiếp theo**: tuổi của hành khách.
- **2 ký tự cuối**: số ghế được xếp.

Trả về **số hành khách có tuổi lớn hơn 60** (nghiêm ngặt: `> 60`, tuổi đúng `60` **không** tính).

| Ví dụ | details | Output | Giải thích |
|---|---|---|---|
| 1 | `["7868190130M7522","5303914400F9211","9273338290F4010"]` | `2` | Tuổi của ba hành khách là `75`, `92`, `40` ⇒ có `2` người trên 60. |
| 2 | `["1313579440F2036","2921522980M5644"]` | `0` | Tuổi là `20` và `56` ⇒ không ai trên 60. |

Constraints:

- `1 <= details.length <= 100`
- `details[i].length == 15`
- `details[i]` chỉ gồm chữ số `'0'`–`'9'`, riêng `details[i][10]` là `'M'`, `'F'` hoặc `'O'`
- Số điện thoại và số ghế của các hành khách **đôi một khác nhau**

In [1]:
d = "7868190130M7522"
print(len(d))
print("điện thoại:", d[0:10])
print("giới tính :", d[10])
print("tuổi      :", d[11:13])
print("ghế       :", d[13:15])

15
điện thoại: 7868190130
giới tính : M
tuổi      : 75
ghế       : 22


## 3. Kiến thức nền

### 3.1. Chuỗi là dãy ký tự, có chỉ số

Chuỗi trong Python là dãy ký tự đánh số từ `0`. `d[0]` là ký tự đầu, `d[10]` là ký tự thứ 11, `d[14]` là ký tự cuối (độ dài `15`). Ký tự đơn lẻ vẫn là một chuỗi độ dài `1`, không phải số.

In [2]:
d = "7868190130M7522"
for i, ch in enumerate(d):
    print(i, repr(ch), end=" | ")
print()
print(type(d[11]), d[11] + d[12])    # <class 'str'> 75: cộng chuỗi là NỐI chuỗi, không phải cộng số

0 '7' | 1 '8' | 2 '6' | 3 '8' | 4 '1' | 5 '9' | 6 '0' | 7 '1' | 8 '3' | 9 '0' | 10 'M' | 11 '7' | 12 '5' | 13 '2' | 14 '2' | 
<class 'str'> 75


### 3.2. Cắt lát `s[a:b]`

`s[a:b]` lấy các ký tự từ chỉ số `a` đến chỉ số `b − 1` (**không gồm `b`**), độ dài là `b − a`. Với tuổi: bắt đầu tại chỉ số `11`, cần `2` ký tự ⇒ `d[11:13]` (chỉ số `11` và `12`).

Cách nhớ: "trái gồm, phải không gồm" — số ký tự = `phải − trái`.

In [3]:
d = "7868190130M7522"
print(d[0:10], len(d[0:10]))       # 10 ký tự: số điện thoại
print(d[11:13], len(d[11:13]))     # 2 ký tự: tuổi
print(d[11:12], len(d[11:12]))     # 1 ký tự: chỉ chữ số hàng chục -> SAI nếu muốn cả tuổi
print(d[13:], d[-2:])              # '22' '22': hai ký tự cuối bằng cắt lát cuối

7868190130 10
75 2
7 1
22 22


### 3.3. `int(chuỗi)`: đổi chuỗi chữ số thành số

`int("75")` cho `75`. Chữ số `0` đứng đầu được bỏ: `int("07") == 7`, `int("00") == 0`. Nếu chuỗi không hợp lệ (có chữ cái, rỗng) thì báo `ValueError`. Dữ liệu của đề đảm bảo tuổi chỉ gồm chữ số nên an toàn.

In [4]:
print(int("75"), int("07"), int("00"), int("60") > 60, int("61") > 60)
try:
    int("M7")
except ValueError as e:
    print("lỗi:", e)

75 7 0 False True
lỗi: invalid literal for int() with base 10: 'M7'


### 3.4. So sánh chuỗi theo thứ tự từ điển

Hai chuỗi được so sánh **từng ký tự từ trái sang phải** theo mã ký tự (`'0' < '1' < ... < '9'`). Nếu hai chuỗi **cùng độ dài** và chỉ gồm chữ số thì so sánh từ điển **trùng** với so sánh số: `"75" > "60"` (`'7' > '6'`), `"60" > "60"` là `False`, `"61" > "60"` (`'6' == '6'` rồi `'1' > '0'`).

**Cảnh báo:** nếu độ dài khác nhau thì sai: `"9" > "60"` là `True` (vì `'9' > '6'`) trong khi `9 < 60`. Ở bài này tuổi luôn đúng 2 ký tự nên an toàn.

In [5]:
print("75" > "60", "60" > "60", "61" > "60", "09" > "60", "99" > "60")
print("9" > "60", 9 > 60)            # True False: khác độ dài thì KHÔNG dùng so sánh chuỗi

True False True False True
True False


### 3.5. Mã ký tự `ord` và biểu thức sinh

- `ord("7")` là mã Unicode của ký tự `'7'` (`55`). Vì `'0'` có mã `48`, chữ số của ký tự `c` là `ord(c) − ord("0")` (= `ord(c) − 48`). Cách này giải thích **vì sao** `int("7")` cho `7`.
- `sum(điều_kiện for d in details)` đếm số phần tử thoả điều kiện (vì `True == 1`, `False == 0`).

In [6]:
print(ord("0"), ord("7"), ord("7") - ord("0"))     # 48 55 7
print(sum(int(d[11:13]) > 60 for d in ["7868190130M7522", "1313579440F2036"]))   # 1

48 55 7
1


### 3.6. Biểu thức chính quy (regex) cơ bản với `re`

`re.fullmatch(mẫu, chuỗi)` kiểm tra **toàn bộ** chuỗi khớp mẫu và trả về đối tượng khớp (hoặc `None`). Trong mẫu: `\d` là một chữ số, `\d{10}` là đúng 10 chữ số, `[MFO]` là một trong ba chữ cái, dấu ngoặc `( )` tạo **nhóm** để lấy phần tương ứng bằng `.group(k)`. Mẫu của đề: `(\d{10})([MFO])(\d{2})(\d{2})`.

In [7]:
import re
m = re.fullmatch(r"(\d{10})([MFO])(\d{2})(\d{2})", "7868190130M7522")
print(m.groups())                   # ('7868190130', 'M', '75', '22')
print(int(m.group(3)))              # 75
print(re.fullmatch(r"(\d{10})([MFO])(\d{2})(\d{2})", "xxx"))   # None

('7868190130', 'M', '75', '22')
75
None


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vì **định dạng cố định**, ta không cần "tìm" tuổi bằng cách phân tích chuỗi phức tạp. Tuổi luôn nằm ở **hai ký tự có chỉ số `11` và `12`**:

```
chỉ số: 0 1 2 3 4 5 6 7 8 9 | 10 | 11 12 | 13 14
chuỗi : 7 8 6 8 1 9 0 1 3 0 |  M |  7  5 |  2  2
         10 ký tự điện thoại | giới | tuổi | ghế
```

Với mỗi hành khách: lấy `d[11:13]`, đổi thành số, so sánh `> 60`; đếm các hành khách thoả mãn.

Tính tay ví dụ 1:

- `"7868190130M7522"` ⇒ tuổi `75` ⇒ `75 > 60` ✓.
- `"5303914400F9211"` ⇒ ký tự `[11:13]`: chuỗi là `5303914400` + `F` + `92` + `11` ⇒ tuổi `92` ✓.
- `"9273338290F4010"` ⇒ tuổi `40` ✗.

Tổng `2`.

In [8]:
details = ["7868190130M7522", "5303914400F9211", "9273338290F4010"]
for d in details:
    age = int(d[11:13])
    print(f"{d} -> tuổi {age:2} -> {'trên 60' if age > 60 else 'không'}")

7868190130M7522 -> tuổi 75 -> trên 60
5303914400F9211 -> tuổi 92 -> trên 60
9273338290F4010 -> tuổi 40 -> không


## 5. Cách 1 — Brute Force / baseline: vòng lặp đếm tay

**Ý tưởng:** biến `count = 0`; với mỗi chuỗi `d` lấy tuổi `age = int(d[11:13])`; nếu `age > 60` thì `count += 1`. Rất trực tiếp.

In [9]:
from typing import List

class SolutionLoop:
    def countSeniors(self, details: List[str]) -> int:
        count = 0
        for d in details:
            age = int(d[11:13])
            if age > 60:
                count += 1
        return count

In [10]:
sol = SolutionLoop()
print(sol.countSeniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"]))   # 2
print(sol.countSeniors(["1313579440F2036", "2921522980M5644"]))                      # 0
print(sol.countSeniors(["1111111111O6000"]))                                         # 0 (đúng 60 không tính)

2
0
0


**Độ phức tạp:** `n = len(details)`. Mỗi chuỗi cắt lát hằng số (2 ký tự) và đổi sang số ⇒ Time `O(n)`; Space `O(1)`.

**Nút thắt:** không có — đã tối ưu về bậc (phải xem mỗi hành khách). Các cải tiến sau chỉ để **gọn hơn** hoặc **tránh `int`**.

## 6. Ý tưởng thuật toán / bước đột phá

Bài này không có "đột phá thuật toán" — nó là bài **xử lý chuỗi cố định**. Điểm đáng học là **ba cách rút gọn** cùng một ý:

1. **Đếm bằng `sum`.** `sum(int(d[11:13]) > 60 for d in details)` thay vòng lặp và biến đếm.
2. **Không cần đổi cả `"75"` thành số.** Chỉ cần so sánh hai chữ số: tuổi `> 60` ⟺ chữ số hàng chục `> 6` **hoặc** (`= 6` **và** hàng đơn vị `> 0`). Hoặc nhờ so sánh từ điển: `d[11:13] > "60"`.
3. **Biến đổi chữ số bằng `ord`** nếu không muốn dùng `int`.

Còn nếu định dạng phức tạp hơn (độ dài thay đổi), nên dùng **regex** để tách thành phần rõ ràng.

In [11]:
for age in ["59", "60", "61", "70", "99", "09"]:
    tens, ones = int(age[0]), int(age[1])
    kq1 = int(age) > 60
    kq2 = tens > 6 or (tens == 6 and ones > 0)
    kq3 = age > "60"
    print(age, kq1, kq2, kq3)
    assert kq1 == kq2 == kq3

59 False False False
60 False False False
61 True True True
70 True True True
99 True True True
09 False False False


## 7. Các cách tiếp theo

### Cách 2 — `sum` với biểu thức sinh

`sum(int(d[11:13]) > 60 for d in details)`. Biểu thức sinh trả `True/False` cho từng chuỗi; `sum` cộng chúng (mỗi `True` là `1`).

In [12]:
class SolutionSum:
    def countSeniors(self, details: List[str]) -> int:
        return sum(int(d[11:13]) > 60 for d in details)

print(SolutionSum().countSeniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"]))   # 2
print(SolutionSum().countSeniors(["1313579440F2036", "2921522980M5644"]))                      # 0

2
0


**Độ phức tạp:** Time `O(n)`; Space `O(1)` (generator không tạo danh sách).

### Cách 3 — Ghép chữ số bằng phép tính, không dùng `int`

Tuổi = `chữ_số_hàng_chục × 10 + chữ_số_hàng_đơn_vị`. Chữ số lấy bằng `ord(ký_tự) − ord("0")`. Cách này lộ rõ "bên trong" của `int(...)`.

In [13]:
class SolutionDigits:
    def countSeniors(self, details: List[str]) -> int:
        count = 0
        for d in details:
            tens = ord(d[11]) - ord("0")
            ones = ord(d[12]) - ord("0")
            if tens * 10 + ones > 60:
                count += 1
        return count

print(SolutionDigits().countSeniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"]))   # 2
print(SolutionDigits().countSeniors(["1313579440F2036", "2921522980M5644"]))                      # 0

2
0


**Độ phức tạp:** Time `O(n)`; Space `O(1)`.

### Cách 4 — So sánh chuỗi: `d[11:13] > "60"`

Vì tuổi luôn **đúng hai chữ số** (có thể có `0` đứng đầu như `"07"`), so sánh từ điển với `"60"` cho đúng kết quả số học (xem 3.4). Không cần `int`.

In [14]:
class SolutionStringCompare:
    def countSeniors(self, details: List[str]) -> int:
        return sum(d[11:13] > "60" for d in details)

print(SolutionStringCompare().countSeniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"]))   # 2
print(SolutionStringCompare().countSeniors(["1313579440F2036", "2921522980M5644"]))                      # 0
print(SolutionStringCompare().countSeniors(["1111111111F6100", "2222222222M6000", "3333333333O0900"]))   # 1

2
0
1


**Độ phức tạp:** Time `O(n)`; Space `O(1)`. **Hạn chế:** chỉ đúng khi mọi tuổi cùng độ dài (ở đây luôn `2`); đây là mẹo, không phải thói quen tốt cho dữ liệu tổng quát.

### Cách 5 — Regex tách thành phần

Dùng `re.fullmatch` với nhóm cho từng thành phần: `(\d{10})([MFO])(\d{2})(\d{2})`. Lấy nhóm thứ ba làm tuổi. Ưu điểm: **mẫu tự mô tả định dạng** và tự kiểm tra tính hợp lệ (trả `None` nếu sai định dạng); nhược điểm: chậm hơn cắt lát.

In [15]:
import re

PATTERN = re.compile(r"(\d{10})([MFO])(\d{2})(\d{2})")

class SolutionRegex:
    def countSeniors(self, details: List[str]) -> int:
        count = 0
        for d in details:
            m = PATTERN.fullmatch(d)
            phone, gender, age, seat = m.groups()
            if int(age) > 60:
                count += 1
        return count

print(SolutionRegex().countSeniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"]))   # 2
print(SolutionRegex().countSeniors(["1313579440F2036", "2921522980M5644"]))                      # 0

2
0


**Độ phức tạp:** Time `O(n)` (mẫu cố định 15 ký tự mỗi chuỗi); Space `O(1)`. Hằng số lớn hơn cắt lát. Hữu ích khi dữ liệu có thể sai định dạng và ta muốn phát hiện.

## 8. TRACE từng bước

Trace Cách 1 trên `["7868190130M7522","5303914400F9211","9273338290F4010"]`:

| `i` | `d` | `d[11:13]` | `age` | `age > 60`? | `count` sau |
|---|---|---|---|---|---|
| 0 | `7868190130M7522` | `"75"` | 75 | có | 1 |
| 1 | `5303914400F9211` | `"92"` | 92 | có | 2 |
| 2 | `9273338290F4010` | `"40"` | 40 | không | 2 |

Kết quả `2`. Dưới đây là hàm trace in cả bốn phần của chuỗi để thấy rõ vì sao `[11:13]` là tuổi.

In [16]:
def trace_seniors(details):
    count = 0
    for i, d in enumerate(details):
        phone, gender, age_s, seat = d[:10], d[10], d[11:13], d[13:]
        age = int(age_s)
        senior = age > 60
        if senior:
            count += 1
        print(f"i={i} d={d} | điện thoại={phone} giới tính={gender} tuổi={age_s}->{age} ghế={seat} | > 60? {senior} | count={count}")
    print("kết quả:", count)
    return count

trace_seniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"])

i=0 d=7868190130M7522 | điện thoại=7868190130 giới tính=M tuổi=75->75 ghế=22 | > 60? True | count=1
i=1 d=5303914400F9211 | điện thoại=5303914400 giới tính=F tuổi=92->92 ghế=11 | > 60? True | count=2
i=2 d=9273338290F4010 | điện thoại=9273338290 giới tính=F tuổi=40->40 ghế=10 | > 60? False | count=2
kết quả: 2


2

In [17]:
# ca biên: đúng 60, và tuổi có số 0 đứng đầu
trace_seniors(["1111111111F6100", "2222222222M6000", "3333333333O0900"])

i=0 d=1111111111F6100 | điện thoại=1111111111 giới tính=F tuổi=61->61 ghế=00 | > 60? True | count=1
i=1 d=2222222222M6000 | điện thoại=2222222222 giới tính=M tuổi=60->60 ghế=00 | > 60? False | count=1
i=2 d=3333333333O0900 | điện thoại=3333333333 giới tính=O tuổi=09->9 ghế=00 | > 60? False | count=1
kết quả: 1


1

## 9. Code nộp LeetCode

Dùng Cách 2: ngắn, rõ, dùng `int` để tránh mẹo so sánh chuỗi.

In [18]:
from typing import List

class Solution:
    def countSeniors(self, details: List[str]) -> int:
        return sum(int(d[11:13]) > 60 for d in details)

In [19]:
print(Solution().countSeniors(["7868190130M7522", "5303914400F9211", "9273338290F4010"]))   # 2
print(Solution().countSeniors(["1313579440F2036", "2921522980M5644"]))                      # 0

2
0


## 10. Vì sao thuật toán đúng

1. **Định vị tuổi.** Theo đề, 10 ký tự đầu là số điện thoại (chỉ số `0..9`), ký tự giới tính ở chỉ số `10`, hai ký tự tuổi ở chỉ số `11` và `12`, hai ký tự ghế ở chỉ số `13`, `14` (tổng `10 + 1 + 2 + 2 = 15`, khớp `len == 15`). Cắt lát `d[11:13]` lấy đúng các chỉ số `11` và `12`.
2. **Đổi sang số.** Tuổi gồm đúng hai chữ số (có thể bắt đầu bằng `0`); `int` cho giá trị số tương ứng, ví dụ `int("07") = 7`.
3. **Điều kiện.** Đề yêu cầu "**lớn hơn** 60" nên điều kiện là `age > 60` — tuổi `60` không được đếm.
4. **Đếm.** Mỗi hành khách đúng một chuỗi; tổng các `True` bằng số hành khách thoả điều kiện.
5. **Cách 4 (so sánh chuỗi).** Hai chuỗi cùng độ dài 2, chỉ gồm chữ số: `a > b` theo từ điển ⟺ chữ số hàng chục của `a` lớn hơn, hoặc bằng nhau và chữ số hàng đơn vị lớn hơn ⟺ `int(a) > int(b)`. Vì vậy `d[11:13] > "60"` ⟺ `int(d[11:13]) > 60`.

## 11. Phân tích Time / Space Complexity

`n = len(details)`; độ dài mỗi chuỗi là hằng số `15`.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp đếm tay | `O(n)` | `O(1)` |
| 2. `sum` + generator | `O(n)` | `O(1)` |
| 3. Ghép chữ số bằng `ord` | `O(n)` | `O(1)` |
| 4. So sánh chuỗi | `O(n)` | `O(1)` |
| 5. Regex | `O(n)` (hằng số lớn hơn) | `O(1)` |

Với `n <= 100` mọi cách đều tức thì. Không thể nhanh hơn `O(n)` vì phải xem mỗi hành khách.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: hai ví dụ đề; tuổi **đúng `60`** (không đếm); `61` (đếm); `99`; `00` và có `0` đứng đầu (`07`); chỉ một hành khách; toàn bộ là người cao tuổi; số điện thoại / ghế chứa chữ số `6` và `0` để chắc không nhầm vị trí.

In [20]:
sols = [SolutionLoop(), SolutionSum(), SolutionDigits(), SolutionStringCompare(), SolutionRegex(), Solution()]
cases = [
    ("ví dụ 1", ["7868190130M7522", "5303914400F9211", "9273338290F4010"], 2),
    ("ví dụ 2", ["1313579440F2036", "2921522980M5644"], 0),
    ("tuổi đúng 60", ["1111111111F6000"], 0),
    ("tuổi 61", ["1111111111F6100"], 1),
    ("tuổi 99", ["1111111111O9900"], 1),
    ("tuổi 00", ["1111111111M0000"], 0),
    ("tuổi 07 (số 0 đứng đầu)", ["1111111111M0799"], 0),
    ("tuổi 59", ["1111111111M5999"], 0),
    ("tất cả cao tuổi", ["1111111111M7011", "2222222222F8022", "3333333333O9033"], 3),
    ("hỗn hợp", ["1111111111M6101", "2222222222F6002", "3333333333O5903", "4444444444M7704"], 2),
    ("phone/ghế nhiều số 6 và 0", ["6666666666M1666", "0000000000F6100"], 1),
]
for name, details, exp in cases:
    for s in sols:
        got = s.countSeniors(details[:])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 2
PASS ví dụ 2 -> 0
PASS tuổi đúng 60 -> 0
PASS tuổi 61 -> 1
PASS tuổi 99 -> 1
PASS tuổi 00 -> 0
PASS tuổi 07 (số 0 đứng đầu) -> 0
PASS tuổi 59 -> 0
PASS tất cả cao tuổi -> 3
PASS hỗn hợp -> 2
PASS phone/ghế nhiều số 6 và 0 -> 1


Chú ý ca `"phone/ghế nhiều số 6 và 0"`: `"6666666666M1666"` có tuổi `16` (không phải người cao tuổi dù điện thoại toàn `6`); `"0000000000F6100"` có tuổi `61` ⇒ đếm `1`.

### 12.2. Phản ví dụ cho các lỗi cắt lát

Ta chứng minh test bắt được các lỗi cắt lát phổ biến (mục 13): dùng `d[11:12]` (một ký tự), dùng `d[10:12]` (lẫn ký tự giới tính), dùng `>=` thay `>`.

In [21]:
def sai_mot_ky_tu(details):
    return sum(int(d[11:12]) > 60 for d in details)         # chỉ chữ số hàng chục: luôn <= 9

def sai_lan_gioi_tinh(details):
    return sum(d[10:12].isdigit() and int(d[10:12]) > 60 for d in details)     # d[10] là chữ cái

def sai_lon_hon_bang(details):
    return sum(int(d[11:13]) >= 60 for d in details)

ex1 = ["7868190130M7522", "5303914400F9211", "9273338290F4010"]
print("một ký tự        :", sai_mot_ky_tu(ex1))       # 0
print("lẫn giới tính    :", sai_lan_gioi_tinh(ex1))   # 0
print("dùng >=, tuổi 60 :", sai_lon_hon_bang(["1111111111F6000"]), "(đúng là 0)")
assert sai_mot_ky_tu(ex1) != Solution().countSeniors(ex1)
assert sai_lon_hon_bang(["1111111111F6000"]) != Solution().countSeniors(["1111111111F6000"])

một ký tự        : 0
lẫn giới tính    : 0
dùng >=, tuổi 60 : 1 (đúng là 0)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu **đúng định dạng và constraints**: `1 <= n <= 100`; mỗi chuỗi 15 ký tự gồm 10 chữ số điện thoại (đôi một khác nhau), giới tính `M/F/O`, tuổi `00..99`, ghế `00..99` (đôi một khác nhau — dùng `random.sample`). Oracle tách bằng các biến riêng rồi so sánh số.

In [22]:
import random
random.seed(2678)

def oracle(details):
    cnt = 0
    for d in details:
        phone = d[0:10]
        gender = d[10]
        age = int(d[11]) * 10 + int(d[12])
        seat = d[13:15]
        assert len(d) == 15 and gender in "MFO" and (phone + seat).isdigit()
        if age > 60:
            cnt += 1
    return cnt

def make_case(n):
    phones = set()
    while len(phones) < n:
        phones.add("".join(random.choice("0123456789") for _ in range(10)))
    seats = random.sample(range(100), n)               # ghế đôi một khác nhau
    out = []
    for phone, seat in zip(phones, seats):
        gender = random.choice("MFO")
        age = random.choice([random.randint(0, 99), random.randint(55, 66)])   # nhiều ca gần 60
        out.append(f"{phone}{gender}{age:02d}{seat:02d}")
    return out

for _ in range(500):
    n = random.randint(1, 20)
    details = make_case(n)
    exp = oracle(details)
    for s in sols:
        got = s.countSeniors(details[:])
        assert got == exp, (type(s).__name__, details, got, exp)
print("500 ca ngẫu nhiên: mọi cách khớp oracle")

details = make_case(100)
assert all(s.countSeniors(details[:]) == oracle(details) for s in sols)
print("n = 100 OK, ví dụ:", details[:2])

500 ca ngẫu nhiên: mọi cách khớp oracle
n = 100 OK, ví dụ: ['3144735339M6613', '1385015380M5897']


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cắt lát sai cận — chỉ lấy một chữ số của tuổi

In [23]:
from typing import List

def sai_mot_chu_so(details: List[str]) -> int:
    return sum(int(d[11:12]) > 60 for d in details)

ex = ["7868190130M7522", "5303914400F9211", "9273338290F4010"]
print("sai :", sai_mot_chu_so(ex))                 # 0: int('7') = 7 không > 60
print("đúng:", Solution().countSeniors(ex))        # 2

sai : 0
đúng: 2


**Vì sao sai:** `d[11:12]` chỉ gồm chỉ số `11` (một ký tự). Tuổi cần **hai** ký tự nên phải `d[11:13]`. Nhớ: `s[a:b]` có độ dài `b − a` và **không** gồm chỉ số `b`.

### Lỗi 2: Dùng `>= 60` thay vì `> 60`

In [24]:
def sai_lon_hon_bang(details: List[str]) -> int:
    return sum(int(d[11:13]) >= 60 for d in details)

edge = ["1111111111F6000", "2222222222M6100"]
print("sai :", sai_lon_hon_bang(edge))             # 2: tuổi 60 bị đếm nhầm
print("đúng:", Solution().countSeniors(edge))      # 1

sai : 2
đúng: 1


**Vì sao sai:** đề viết "**strictly more than 60 years old**" — nghiêm ngặt. Người đúng `60` tuổi **không** được đếm. Mỗi khi thấy "hơn / nhiều hơn", dùng `>`; "ít nhất / không nhỏ hơn" mới dùng `>=`.

### Lỗi 3: So sánh chuỗi với số (hoặc quên đổi `int`)

In [25]:
def sai_so_sanh_kieu(details: List[str]) -> int:
    count = 0
    for d in details:
        try:
            if d[11:13] > 60:        # chuỗi so với số
                count += 1
        except TypeError as e:
            return f"TypeError: {e}"
    return count

print("sai :", sai_so_sanh_kieu(["7868190130M7522"]))
print("đúng:", Solution().countSeniors(["7868190130M7522"]))

sai : TypeError: '>' not supported between instances of 'str' and 'int'
đúng: 1


**Vì sao sai:** Python 3 **không** so sánh được chuỗi với số (`'75' > 60` báo `TypeError`). Hoặc đổi `int(...)`, hoặc so sánh với chuỗi `"60"` (và chỉ khi các chuỗi cùng độ dài).

### Lỗi 4: Nhầm vị trí — lẫn ký tự giới tính hoặc lấy ghế làm tuổi

In [26]:
def sai_lay_ghe(details: List[str]) -> int:
    return sum(int(d[13:15]) > 60 for d in details)          # lấy hai ký tự cuối: đó là ghế

def sai_lay_chi_so_cu(details: List[str]) -> int:
    return sum(int(d[10:12]) > 60 for d in details if d[10].isdigit())   # d[10] là chữ cái -> bị bỏ hết

ex = ["7868190130M7522", "5303914400F9211", "9273338290F4010"]
print("lấy ghế     :", sai_lay_ghe(ex))           # 0 (ghế là 22, 11, 10)
print("lấy chỉ số 10:", sai_lay_chi_so_cu(ex))     # 0
print("đúng        :", Solution().countSeniors(ex))

lấy ghế     : 0
lấy chỉ số 10: 0
đúng        : 2


**Vì sao sai:** định dạng là `10 + 1 + 2 + 2`. Tuổi bắt đầu **sau** ký tự giới tính (chỉ số `10`), tức tại chỉ số `11`. Cách tránh: vẽ bảng chỉ số như mục 4 hoặc in ra `d[0:10]`, `d[10]`, `d[11:13]`, `d[13:]` để kiểm tra bằng mắt.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chuỗi định dạng **cố định** ⇒ cắt lát theo chỉ số đã biết; vẽ bảng "chỉ số — ý nghĩa" trước khi code.
- Quy tắc cắt lát `s[a:b]`: độ dài `b − a`, **không gồm `b`**. Tuổi: bắt đầu `11`, dài `2` ⇒ `[11:13]`.
- Đếm theo điều kiện ⇒ `sum(điều_kiện for x in ...)`.
- "Strictly more than" ⇒ `>`; "at least" ⇒ `>=`. Luôn thử một giá trị **đúng bằng ngưỡng** (tuổi `60`).
- Đổi chữ số sang số bằng `int(...)`; so sánh chuỗi số chỉ an toàn khi cùng độ dài.
- Dữ liệu không đảm bảo định dạng ⇒ dùng **regex** hoặc kiểm tra `len`, `isdigit` trước.

## 15. Biến thể của bài

1. **Đếm người từ 18 đến 60 tuổi** (hai đầu gồm cả) — thêm điều kiện kép `18 <= age <= 60`.
2. **Đếm theo giới tính** của người cao tuổi — dùng `Counter` hoặc `dict`.
3. **Tuổi trung bình** của mọi hành khách — `sum(ages) / len(ages)`.
4. **Ghế được xếp** của người cao tuổi: lấy `d[13:15]` rồi sắp xếp.
5. **Bài xử lý chuỗi định dạng khác:** Validate IP Address (LeetCode 468), Number of Valid Words (ký tự đặc biệt), Reformat Phone Number (LeetCode 1694).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp | `int(d[11:13]) > 60`, đếm tay | `O(n)` | `O(1)` | Rõ ràng cho người mới |
| 2. `sum` + generator | Cùng ý, gọn | `O(n)` | `O(1)` | **Nên nộp** |
| 3. Ghép chữ số bằng `ord` | Tự làm phép của `int` | `O(n)` | `O(1)` | Hiểu bản chất; hiếm khi cần |
| 4. So sánh chuỗi `> "60"` | Cùng độ dài ⇒ từ điển = số | `O(n)` | `O(1)` | Mẹo; chỉ đúng khi cùng độ dài |
| 5. Regex | Mẫu mô tả định dạng, tách nhóm | `O(n)` (hằng số lớn) | `O(1)` | Khi định dạng phức tạp / cần kiểm tra |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"7868190130M7522"[11:13]`, `len("7868190130M7522")`, `"75" > "60"`, `int("07")`.

**Bài 2 (tính tay).** Với `details = ["1234567890F6122", "0987654321O6011", "1111111111M0033"]`, ghi tuổi từng người và kết quả của `countSeniors`.

**Bài 3 (điền TODO).** Hoàn thành `get_age_todo(d)` trả tuổi (kiểu `int`) của chuỗi `d`.

In [27]:
def get_age_todo(d):
    # TODO 1: cắt lát hai ký tự tuổi: chỉ số 11 và 12
    # TODO 2: đổi sang int và return
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_seniors_regex(details)` bằng `re.fullmatch` và nhóm, mà không nhìn Cách 5. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_adults(details)` đếm hành khách có `18 <= tuổi <= 60`. (b) Viết `seniors_by_gender(details)` trả từ điển `{'M': ..., 'F': ..., 'O': ...}` số người **trên 60** theo giới tính (có đủ cả ba khoá, kể cả `0`). (c) Viết `average_age(details)` (số thực).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: tuổi nằm ở chỉ số `11, 12`; độ dài `15`; `'7' > '6'` ⇒ `True`; `int("07") = 7`. Bài 2: ba tuổi là `61`, `60`, `00`.
- **Mức 2 (vừa):** Bài 3: `int(d[11:13])`. Bài 4: mẫu `(\d{10})([MFO])(\d{2})(\d{2})`, `m.groups()`, lấy nhóm thứ ba.
- **Mức 3 (sâu):** Bài 5(a): `18 <= age <= 60` (chuỗi so sánh liên tiếp). Bài 5(b): khởi tạo `{'M': 0, 'F': 0, 'O': 0}` rồi `result[d[10]] += 1` khi `age > 60`. Bài 5(c): `sum(ages) / len(ages)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "7868190130M7522"[11:13] == "75"
assert len("7868190130M7522") == 15
assert ("75" > "60") is True
assert int("07") == 7

# Bài 2
details = ["1234567890F6122", "0987654321O6011", "1111111111M0033"]
assert [int(d[11:13]) for d in details] == [61, 60, 0]
assert Solution().countSeniors(details) == 1

# Bài 3
def get_age_ans(d):
    return int(d[11:13])

assert get_age_ans("7868190130M7522") == 75
assert get_age_ans("1111111111M0799") == 7

# Bài 4
import re

def count_seniors_regex(details):
    pat = re.compile(r"(\d{10})([MFO])(\d{2})(\d{2})")
    total = 0
    for d in details:
        m = pat.fullmatch(d)
        if m and int(m.group(3)) > 60:
            total += 1
    return total

assert count_seniors_regex(["7868190130M7522", "5303914400F9211", "9273338290F4010"]) == 2
assert count_seniors_regex(["1313579440F2036", "2921522980M5644"]) == 0

# Bài 5
def count_adults(details):
    return sum(18 <= int(d[11:13]) <= 60 for d in details)

def seniors_by_gender(details):
    result = {"M": 0, "F": 0, "O": 0}
    for d in details:
        if int(d[11:13]) > 60:
            result[d[10]] += 1
    return result

def average_age(details):
    ages = [int(d[11:13]) for d in details]
    return sum(ages) / len(ages)

sample = ["7868190130M7522", "5303914400F9211", "9273338290F4010"]
assert count_adults(sample) == 1                                     # chỉ tuổi 40
assert seniors_by_gender(sample) == {"M": 1, "F": 1, "O": 0}
assert abs(average_age(sample) - (75 + 92 + 40) / 3) < 1e-9
assert count_adults(["1111111111M1800", "2222222222F6000", "3333333333O1700", "4444444444M6100"]) == 2
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tuổi ở `d[11:13]`, đếm bằng `sum`)

```python
from typing import List

class Solution:
    def countSeniors(self, details: List[str]) -> int:
        # chuỗi: 10 ký tự điện thoại | 1 giới tính | 2 tuổi | 2 ghế
        return sum(int(d[11:13]) > 60 for d in details)
```

**Checklist**

- [ ] Tuổi nằm ở `d[11:13]` (chỉ số `11` và `12`; `13` không gồm)?
- [ ] Đã `int(...)` trước khi so sánh số (hoặc so với chuỗi `"60"` cùng độ dài)?
- [ ] Điều kiện là `> 60` (nghiêm ngặt), tuổi `60` không tính?
- [ ] Không nhầm với ký tự giới tính (chỉ số `10`) hoặc ghế (chỉ số `13`, `14`)?
- [ ] Đã thử tuổi `60`, `61`, `00`, `07`, `99`?
- [ ] Signature đúng `countSeniors(self, details: List[str]) -> int` và có `from typing import List`?